CVNN, good for data with phases

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# ==========================================
# GLOBAL HYPERPARAMETERS
# ==========================================
MODEL_TYPE = 'prototype'      # Options: 'concat', 'prototype'
ACTIVATION_TYPE = 'cardioid'  # Options: 'modrelu', 'cardioid'
ENCODING_TYPE = 'spatial'     # Options: 'fft', 'spatial', 'none'
EPOCHS = 20
DROPOUT_P = 0.2               # Complex Dropout probability
# ==========================================

# --- 1. Complex Initialization ---
def complex_xavier_init(module):
    """Initializes complex linear layers by splitting variance across real and imaginary parts."""
    if isinstance(module, nn.Linear) and module.weight.is_complex():
        fan_in, fan_out = nn.init._calculate_fan_in_and_fan_out(module.weight)
        var = 1.0 / (fan_in + fan_out)
        std = math.sqrt(var)
        
        with torch.no_grad():
            module.weight.real.normal_(0, std)
            module.weight.imag.normal_(0, std)
            if module.bias is not None:
                module.bias.zero_()

# --- 2. Encoders & Regularization ---
class InputEncoder(nn.Module):
    def __init__(self, encoding_type=ENCODING_TYPE, size=28):
        super().__init__()
        self.encoding_type = encoding_type
        
        if self.encoding_type == 'spatial':
            coords = torch.linspace(-1, 1, steps=size)
            grid_y, grid_x = torch.meshgrid(coords, coords, indexing='ij')
            spatial_phase = torch.atan2(grid_y, grid_x)
            self.register_buffer('phase_grid', torch.complex(torch.cos(spatial_phase), torch.sin(spatial_phase)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.encoding_type == 'fft':
            z = torch.fft.fft2(x, norm="ortho")
            z = torch.fft.fftshift(z, dim=(-2, -1))
        elif self.encoding_type == 'spatial':
            z = torch.complex(x, torch.zeros_like(x)) * self.phase_grid
        else:
            z = torch.complex(x, torch.zeros_like(x))
            
        return z.view(x.size(0), -1)

class ComplexDropout(nn.Module):
    """Drops entire complex neurons by applying the same mask to real and imaginary parts."""
    def __init__(self, p=DROPOUT_P):
        super().__init__()
        self.p = p

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0.0:
            return z
        mask = F.dropout(torch.ones_like(z.real), p=self.p, training=True)
        return z * mask

# --- 3. Activations ---
class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.full((features,), 0.1))

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        mag = torch.abs(z)
        phase = z / (mag + 1e-5) 
        mag_activated = F.relu(mag + self.b)
        return torch.complex(mag_activated, torch.zeros_like(mag_activated)) * phase

class ComplexCardioid(nn.Module):
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        phase = torch.angle(z)
        scale = 0.5 * (1.0 + torch.cos(phase))
        return z * torch.complex(scale, torch.zeros_like(scale))

# --- 4. Unified Model ---
class CVNN(nn.Module):
    def __init__(self, model_type=MODEL_TYPE, activation_type=ACTIVATION_TYPE, encoding_type=ENCODING_TYPE):
        super().__init__()
        self.model_type = model_type
        self.encoder = InputEncoder(encoding_type)
        self.fc1 = nn.Linear(28 * 28, 128, dtype=torch.complex64)
        self.fc2 = nn.Linear(128, 64, dtype=torch.complex64)
        
        self.act1 = ModReLU(128) if activation_type == 'modrelu' else ComplexCardioid()
        self.act2 = ModReLU(64) if activation_type == 'modrelu' else ComplexCardioid()
        self.dropout = ComplexDropout(p=DROPOUT_P)
        
        if self.model_type == 'concat':
            self.fc3_real = nn.Linear(128, 10, dtype=torch.float32)
        elif self.model_type == 'prototype':
            self.fc_final = nn.Linear(64, 16, dtype=torch.complex64)
            self.class_prototypes = nn.Parameter(torch.randn(10, 16, dtype=torch.complex64))
            # Temperature scaling for bounded cosine similarities
            self.temperature = nn.Parameter(torch.tensor(10.0))
        
        self.apply(complex_xavier_init)
        
        if self.model_type == 'concat':
            nn.init.kaiming_uniform_(self.fc3_real.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor, capture_activations=False):
        z = self.encoder(x)
        
        z1 = self.dropout(self.act1(self.fc1(z)))
        z2 = self.dropout(self.act2(self.fc2(z1)))
        
        if self.model_type == 'concat':
            z_concat = torch.cat([z2.real, z2.imag], dim=1)
            out = self.fc3_real(z_concat)
            
        elif self.model_type == 'prototype':
            z_out = self.fc_final(z2)
            
            # Hermitian Inner Product: Re(Z * P^*)
            inner_product = torch.matmul(z_out, self.class_prototypes.conj().T).real
            
            # Magnitudes
            z_norm = torch.linalg.norm(z_out, dim=1, keepdim=True)
            p_norm = torch.linalg.norm(self.class_prototypes, dim=1).unsqueeze(0)
            
            # Hermitian Cosine Similarity
            denom = torch.clamp(z_norm * p_norm, min=1e-5)
            sim = inner_product / denom
            
            # Scale to logits
            out = sim * self.temperature
        
        if capture_activations:
            return out, z1
        return out

# --- 5. Training Engine ---
def train_model(model, file_name, epochs=EPOCHS):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = transforms.Compose([transforms.ToTensor()])
    train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)
    test_loader = DataLoader(datasets.MNIST('./data', train=False, transform=transform), batch_size=1000, shuffle=False)

    model = model.to(device)
    lr = 1e-2 if ACTIVATION_TYPE == 'cardioid' else 1e-3
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        w_mag = torch.abs(model.fc1.weight)
        topk_idx = torch.topk(w_mag.flatten(), 50).indices
        row_idx = topk_idx // model.fc1.weight.shape[1]
        col_idx = topk_idx % model.fc1.weight.shape[1]

    history = {
        'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
        'phases': [], 'magnitudes': [], 'tracked_weights': [],
        'grad_real_norm': [], 'grad_imag_norm': []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct_train, total_train = 0.0, 0, 0
        epoch_grad_real, epoch_grad_imag = 0.0, 0.0
        
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            
            if model.fc1.weight.grad is not None:
                epoch_grad_real += model.fc1.weight.grad.real.norm().item()
                epoch_grad_imag += model.fc1.weight.grad.imag.norm().item()
                
            optimizer.step()
            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct_train += pred.eq(target.view_as(pred)).sum().item()
            total_train += data.size(0)

        train_acc = correct_train / total_train
        history['grad_real_norm'].append(epoch_grad_real / len(train_loader))
        history['grad_imag_norm'].append(epoch_grad_imag / len(train_loader))

        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        all_phases, all_mags = [], []
        
        with torch.no_grad():
            for i, (data, target) in enumerate(test_loader):
                data, target = data.to(device), target.to(device)
                output, z1 = model(data, capture_activations=True)
                loss = criterion(output, target)
                val_loss += loss.item() * data.size(0)
                pred = output.argmax(dim=1, keepdim=True)
                correct_val += pred.eq(target.view_as(pred)).sum().item()
                total_val += data.size(0)
                
                if i == 0:
                    all_phases.append(torch.angle(z1).cpu())
                    all_mags.append(torch.abs(z1).cpu())

            history['tracked_weights'].append(model.fc1.weight[row_idx, col_idx].cpu().clone())

        val_acc = correct_val / total_val
        history['train_loss'].append(total_loss / total_train)
        history['val_loss'].append(val_loss / total_val)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['phases'].append(torch.cat(all_phases).flatten().numpy())
        history['magnitudes'].append(torch.cat(all_mags).flatten().numpy())

        scheduler.step()

        print(f"Epoch {epoch:02d} | Train Loss: {history['train_loss'][-1]:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    torch.save(history, file_name)
    print(f"Saved {file_name}")

# --- 6. Execution ---
if __name__ == "__main__":
    file_name = f'cvnn_telemetry.pt'
    print(f"Training CVNN (Model: {MODEL_TYPE}, Act: {ACTIVATION_TYPE}, Enc: {ENCODING_TYPE}, Epochs: {EPOCHS}, Dropout: {DROPOUT_P})...")
    train_model(CVNN(), file_name)

# --- 7. Plotting Dashboard ---
def plot_dashboard(file_name='cvnn_telemetry.pt'):
    hist = torch.load(file_name, weights_only=False)
    epochs = len(hist['train_loss'])

    def render_epoch(epoch):
        idx = epoch - 1
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        
        ax = axes[0, 0]
        ax.plot(range(1, epochs + 1), hist['train_acc'], label='Train Acc', color='blue')
        ax.plot(range(1, epochs + 1), hist['val_acc'], label='Val Acc', color='orange', linestyle='--')
        ax.axvline(x=epoch, color='red', alpha=0.3)
        ax.set_title("Generalization Gap")
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        ax = axes[0, 1]
        ax.plot(range(1, epochs + 1), hist['train_loss'], label='Train Loss', color='blue')
        ax.plot(range(1, epochs + 1), hist['val_loss'], label='Val Loss', color='orange', linestyle='--')
        ax.axvline(x=epoch, color='red', alpha=0.3)
        ax.set_title("Loss Convergence")
        ax.legend()
        ax.grid(True, alpha=0.3)

        ax = axes[0, 2]
        ax.plot(range(1, epochs + 1), hist['grad_real_norm'], label='Real Grad', color='green')
        ax.plot(range(1, epochs + 1), hist['grad_imag_norm'], label='Imag Grad', color='purple')
        ax.axvline(x=epoch, color='red', alpha=0.3)
        ax.set_title("Gradient Norms (Layer 1)")
        ax.legend()
        ax.grid(True, alpha=0.3)

        ax = axes[1, 0]
        w = hist['tracked_weights'][idx].numpy()
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.scatter(w.real, w.imag, c='red', alpha=0.6)
        limit = np.max(np.abs(torch.stack(hist['tracked_weights']).numpy())) * 1.1
        ax.set_xlim(-limit, limit)
        ax.set_ylim(-limit, limit)
        ax.set_title(f"Tracked Weights (Epoch {epoch})")
        
        ax = axes[1, 1]
        ax.hist(hist['phases'][idx], bins=50, color='teal', alpha=0.7, density=True)
        ax.axhline(1 / (2 * np.pi), color='red', linestyle='dashed')
        ax.set_xlim(-np.pi, np.pi)
        ax.set_title(f"Phase Distribution (Epoch {epoch})")
        
        ax = axes[1, 2]
        ax.hist(hist['magnitudes'][idx], bins=50, color='orange', alpha=0.7, density=True)
        ax.set_title(f"Magnitude Distribution (Epoch {epoch})")

        plt.tight_layout()
        plt.show()

    interact(render_epoch, epoch=IntSlider(min=1, max=epochs, step=1, value=1, description='Epoch:'))

Training CVNN (Model: prototype, Act: cardioid, Enc: spatial, Epochs: 20, Dropout: 0.2)...
Epoch 01 | Train Loss: 0.2684 | Train Acc: 0.9174 | Val Acc: 0.9622 | LR: 0.009938
Epoch 02 | Train Loss: 0.1290 | Train Acc: 0.9607 | Val Acc: 0.9700 | LR: 0.009755
Epoch 03 | Train Loss: 0.1001 | Train Acc: 0.9686 | Val Acc: 0.9736 | LR: 0.009455
Epoch 04 | Train Loss: 0.0817 | Train Acc: 0.9744 | Val Acc: 0.9778 | LR: 0.009045
Epoch 05 | Train Loss: 0.0685 | Train Acc: 0.9784 | Val Acc: 0.9756 | LR: 0.008536
Epoch 06 | Train Loss: 0.0615 | Train Acc: 0.9803 | Val Acc: 0.9774 | LR: 0.007939
Epoch 07 | Train Loss: 0.0537 | Train Acc: 0.9830 | Val Acc: 0.9788 | LR: 0.007270
Epoch 08 | Train Loss: 0.0449 | Train Acc: 0.9855 | Val Acc: 0.9822 | LR: 0.006545
Epoch 09 | Train Loss: 0.0392 | Train Acc: 0.9870 | Val Acc: 0.9793 | LR: 0.005782
Epoch 10 | Train Loss: 0.0312 | Train Acc: 0.9898 | Val Acc: 0.9812 | LR: 0.005000
Epoch 11 | Train Loss: 0.0287 | Train Acc: 0.9906 | Val Acc: 0.9830 | LR: 0.004

In [ ]:
plot_dashboard('cvnn_telemetry.pt')


interactive(children=(IntSlider(value=1, description='Epoch:', max=20, min=1), Output()), _dom_classes=('widge…